# Fase 3 — Regras de Associação: Candidatos 2026 (TSE)

## Pergunta de negócio

> **Que combinações de características dos candidatos aparecem juntas com mais frequência do que se fossem independentes? Em particular: quais atributos (partido, gênero, escolaridade, cor/raça, UF de nascimento) estão associados a cada perfil identificado na Fase 2 — e, assim que a eleição ocorrer, a ser eleito?**

Até aqui, a clusterização (Fase 2) agrupou candidatos por **distância** em variáveis numéricas (idade, escolaridade, patrimônio). Regras de associação fazem uma pergunta diferente: em vez de "quem está perto de quem", **quais combinações de atributos categóricos aparecem juntas com mais frequência do que o acaso explicaria**? A técnica clássica pra isso é o **Apriori**, que originalmente resolve o problema da "cesta de compras" (que produtos são comprados juntos) — aqui, cada candidato é uma cesta, e cada valor de cada variável categórica (partido, gênero, escolaridade...) é um item dessa cesta.


## Configuração

Use os **mesmos valores** que você usou nas Fases 0/1/2, para carregar o arquivo certo.


In [30]:
CARGO = "DEPUTADO FEDERAL"   # mesmo valor usado nas Fases 0/1/2
UF = 'AL', 'BA', 'CE', 'MA', 'PB', 'PE', 'PI', 'RN', 'SE'               # mesmo valor usado nas Fases 0/1/2
ANO_ELEICAO = 2026


## Preparando a base — reconstituindo os clusters nomeados da Fase 2

Apriori precisa de **itens categóricos**, não de distância — mas queremos incluir o cluster da Fase 2 como mais um item da cesta (pra achar regras do tipo "esse perfil de candidato é dos Abastados"). Na Fase 2 (`02_clusterizacao.ipynb`), o **Bisecting K-Means com k=3** (critério de inércia média) foi o resultado final escolhido, e os três clusters foram batizados a partir da ficha técnica e da composição demográfica/partidária:

- **Abastados** — 78% dos candidatos, os únicos com algum patrimônio declarado.
- **Jovens** — o grupo mais novo (39 anos em média), quase sem patrimônio declarado.
- **Sem bens e alta escolaridade** — 100% sem patrimônio declarado, a maior escolaridade média dos três (e também o grupo mais velho).

Recalculamos aqui, rapidamente, a mesma partição (mesma `SEMENTE`, mesmo critério, mesmos nomes) — mesma lógica de auto-suficiência já usada no `02_clusterizacao_outros.ipynb`.


In [31]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from mlxtend.frequent_patterns import apriori, association_rules
from pyvis.network import Network
from IPython.display import IFrame

sns.set_style("whitegrid")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)  # não truncar antecedents/consequents nas tabelas

SEMENTE = 42
np.random.seed(SEMENTE)  # mesmo racional de reprodutibilidade defensiva dos notebooks anteriores

nome_uf = UF if UF is not None else 'BRASIL'
nome_cargo = CARGO.replace(' ', '_')
arquivo = f"dados/candidatos_{nome_cargo}_nordeste_{ANO_ELEICAO}.csv"

df = pd.read_csv(arquivo)
print(f"Carregado: {arquivo}  ->  {df.shape[0]} candidatos, {df.shape[1]} colunas")
df.head()


Carregado: dados/candidatos_DEPUTADO_FEDERAL_nordeste_2026.csv  ->  2189 candidatos, 59 colunas


,DT_GERACAO,HH_GERACAO,ANO_ELEICAO,CD_TIPO_ELEICAO,NM_TIPO_ELEICAO,NR_TURNO,CD_ELEICAO,DS_ELEICAO,DT_ELEICAO,TP_ABRANGENCIA,SG_UF,SG_UE,NM_UE,CD_CARGO,DS_CARGO,SQ_CANDIDATO,NR_CANDIDATO,NM_CANDIDATO,NM_URNA_CANDIDATO,NM_SOCIAL_CANDIDATO,NR_CPF_CANDIDATO,DS_EMAIL,CD_SITUACAO_CANDIDATURA,DS_SITUACAO_CANDIDATURA,TP_AGREMIACAO,NR_PARTIDO,SG_PARTIDO,NM_PARTIDO,NR_FEDERACAO,NM_FEDERACAO,SG_FEDERACAO,DS_COMPOSICAO_FEDERACAO,SQ_COLIGACAO,NM_COLIGACAO,DS_COMPOSICAO_COLIGACAO,SG_UF_NASCIMENTO,DT_NASCIMENTO,NR_TITULO_ELEITORAL_CANDIDATO,CD_GENERO,DS_GENERO,CD_GRAU_INSTRUCAO,DS_GRAU_INSTRUCAO,CD_ESTADO_CIVIL,DS_ESTADO_CIVIL,CD_COR_RACA,DS_COR_RACA,CD_OCUPACAO,DS_OCUPACAO,CD_SIT_TOT_TURNO,DS_SIT_TOT_TURNO,BENS IMÓVEIS,BENS MÓVEIS,DINHEIRO,INVESTIMENTOS RENDA FIXA,INVESTIMENTOS RENDA VARIÁVEL,OUTROS BENS,Total_Bens,Total_Bens_Log,IDADE
0,23/08/2026,19:30:42,2026,2,ELEIÇÃO ORDINÁRIA,1,6259,Eleições Gerais Estaduais 2026,2026-10-04,ESTADUAL,AL,AL,ALAGOAS,6,DEPUTADO FEDERAL,20002532430,3033,CAROLAYNNE PIRES DE OLIVEIRA SOUZA,CAROL SOUZA,#NULO,10279565488,NÃO DIVULGÁVEL,-3,#NE,PARTIDO ISOLADO,30,NOVO,PARTIDO NOVO,-1,#NULO,#NULO,#NULO,20001800111,PARTIDO ISOLADO,NOVO,AL,1991-11-10,40248081791,4,FEMININO,8,SUPERIOR COMPLETO,1,SOLTEIRO(A),2,PRETA,132,PSICÓLOGO,-1,#NULO,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.000000,34
1,23/08/2026,19:30:42,2026,2,ELEIÇÃO ORDINÁRIA,1,6259,Eleições Gerais Estaduais 2026,2026-10-04,ESTADUAL,AL,AL,ALAGOAS,6,DEPUTADO FEDERAL,20002532431,3030,JOSÉ GUIDO DO RÊGO SANTOS NETO,GUIDO SANTOS,#NULO,5955901442,NÃO DIVULGÁVEL,-3,#NE,PARTIDO ISOLADO,30,NOVO,PARTIDO NOVO,-1,#NULO,#NULO,#NULO,20001800111,PARTIDO ISOLADO,NOVO,AL,1985-06-07,31724561767,2,MASCULINO,8,SUPERIOR COMPLETO,3,CASADO(A),1,BRANCA,296,SERVIDOR PÚBLICO FEDERAL,-1,#NULO,0.0,0.0,45000.00,0.0,0.0,70000.0,115000.00,11.652696,41
2,23/08/2026,19:30:42,2026,2,ELEIÇÃO ORDINÁRIA,1,6259,Eleições Gerais Estaduais 2026,2026-10-04,ESTADUAL,AL,AL,ALAGOAS,6,DEPUTADO FEDERAL,20002532432,3003,JOÃO CARLOS LIMA UCHÔA,JOÃO UCHÔA,#NULO,48288586449,NÃO DIVULGÁVEL,-3,#NE,PARTIDO ISOLADO,30,NOVO,PARTIDO NOVO,-1,#NULO,#NULO,#NULO,20001800111,PARTIDO ISOLADO,NOVO,AL,1965-02-12,5104321775,2,MASCULINO,8,SUPERIOR COMPLETO,3,CASADO(A),1,BRANCA,131,ADVOGADO,-1,#NULO,1850000.0,841018.0,77464.01,0.0,0.0,1000000.0,3768482.01,15.142183,61
3,23/08/2026,19:30:42,2026,2,ELEIÇÃO ORDINÁRIA,1,6259,Eleições Gerais Estaduais 2026,2026-10-04,ESTADUAL,AL,AL,ALAGOAS,6,DEPUTADO FEDERAL,20002532433,3000,ANA CAROLINA BELTRÃO PEIXOTO,PROFESSORA CAROL,#NULO,2734264439,NÃO DIVULGÁVEL,-3,#NE,PARTIDO ISOLADO,30,NOVO,PARTIDO NOVO,-1,#NULO,#NULO,#NULO,20001800111,PARTIDO ISOLADO,NOVO,AL,1978-09-21,24962281775,4,FEMININO,8,SUPERIOR COMPLETO,5,VIÚVO(A),1,BRANCA,297,SERVIDOR PÚBLICO ESTADUAL,-1,#NULO,570000.0,60000.0,0.00,0.0,0.0,0.0,630000.00,13.353477,48
4,23/08/2026,19:30:42,2026,2,ELEIÇÃO ORDINÁRIA,1,6259,Eleições Gerais Estaduais 2026,2026-10-04,ESTADUAL,AL,AL,ALAGOAS,6,DEPUTADO FEDERAL,20002532434,3001,JOSÉ ALEX BRANDÃO SILVEIRA,ALEX BRANDÃO,#NULO,3732710424,NÃO DIVULGÁVEL,-3,#NE,PARTIDO ISOLADO,30,NOVO,PARTIDO NOVO,-1,#NULO,#NULO,#NULO,20001800111,PARTIDO ISOLADO,NOVO,AL,1981-08-03,26986421767,2,MASCULINO,8,SUPERIOR COMPLETO,3,CASADO(A),1,BRANCA,403,"CORRETOR DE IMÓVEIS, SEGUROS, TÍTULOS E VALORES",-1,#NULO,950000.0,0.0,0.00,0.0,0.0,142000.0,1092000.00,13.903522,45


In [32]:
ANOS_ESTUDO_POR_GRAU = {
    'ANALFABETO': 0,
    'LÊ E ESCREVE': 1,
    'ENSINO FUNDAMENTAL INCOMPLETO': 4,
    'ENSINO FUNDAMENTAL COMPLETO': 8,
    'ENSINO MÉDIO INCOMPLETO': 9,
    'ENSINO MÉDIO COMPLETO': 11,
    'SUPERIOR INCOMPLETO': 13,
    'SUPERIOR COMPLETO': 16,
    # 'NÃO DIVULGÁVEL' fica de fora de propósito — mesma decisão da Versão 2 do notebook de clusterização
}

df['ANOS_ESTUDO'] = df['DS_GRAU_INSTRUCAO'].map(ANOS_ESTUDO_POR_GRAU)


In [33]:
colunas_driver = ['IDADE', 'ANOS_ESTUDO', 'Total_Bens_Log']

Q1, Q3 = df['Total_Bens_Log'].quantile([0.25, 0.75])
IQR = Q3 - Q1
limite_superior = Q3 + 1.5 * IQR

df_cluster = df[(df['Total_Bens_Log'] <= limite_superior) & (df['ANOS_ESTUDO'].notna())].copy()
print(f"Base para clusterização/regras: {df_cluster.shape[0]} candidatos "
      f"(descartados {df.shape[0] - df_cluster.shape[0]} por patrimônio extremo ou escolaridade não divulgada)")

scaler = MinMaxScaler()
X = scaler.fit_transform(df_cluster[colunas_driver])


Base para clusterização/regras: 2189 candidatos (descartados 0 por patrimônio extremo ou escolaridade não divulgada)


In [34]:
def inercia(indices):
    if len(indices) < 2:
        return 0.0
    pontos = X[indices]
    centro = pontos.mean(axis=0)
    return float(((pontos - centro) ** 2).sum(axis=1).mean())


def kmeans_k(X, k_max, random_state=SEMENTE):
    """Mesma lógica do 02_clusterizacao.ipynb (Parte 2, critério de inércia média) — mas só
    devolve a partição final em k_max clusters, sem guardar o histórico de divisões (essa
    exploração já foi feita e documentada lá)."""
    clusters = {0: np.arange(X.shape[0])}
    proximo_id = 1
    for _ in range(1, k_max):
        id_escolhido = max(clusters, key=lambda cid: inercia(clusters[cid]))
        indices_pai = clusters[id_escolhido]
        if len(indices_pai) < 2:
            break
        km2 = KMeans(n_clusters=2, random_state=random_state, n_init=10)
        labels2 = km2.fit_predict(X[indices_pai])
        id_a, id_b = proximo_id, proximo_id + 1
        proximo_id += 2
        del clusters[id_escolhido]
        clusters[id_a] = indices_pai[labels2 == 0]
        clusters[id_b] = indices_pai[labels2 == 1]
    return clusters


k = 4  # mesmo k escolhido no 02_clusterizacao.ipynb (Parte 2)

clusters_finais = kmeans_k(X, k)
ids_por_tamanho = sorted(clusters_finais, key=lambda cid: len(clusters_finais[cid]), reverse=True)

rotulos = np.empty(X.shape[0], dtype=object)
for letra, cid in zip([chr(65 + i) for i in range(len(ids_por_tamanho))], ids_por_tamanho):
    rotulos[clusters_finais[cid]] = letra
df_cluster['cluster_kmeans'] = rotulos

# mesmos nomes definidos no 02_clusterizacao.ipynb, a partir da mesma ficha técnica
NOMES_CLUSTER = {
    'A': 'Elite',
    'B': 'Profissionais Diplomados',
    'C': 'Pequenos Empreendedores',
    'D': 'Base Popular'
}
df_cluster['cluster'] = df_cluster['cluster_kmeans'].map(NOMES_CLUSTER)

df_cluster['cluster'].value_counts()


cluster
Elite                       962
Profissionais Diplomados    446
Pequenos Empreendedores     439
Base Popular                342
Name: count, dtype: int64

## Montando a "cesta" de itens

Apriori não trabalha com números contínuos como o K-Means — ele precisa de **itens binários** (presente/ausente), do mesmo jeito que "pão" e "leite" são itens numa cesta de supermercado. Cada linha vira uma "cesta" (um candidato), e cada valor de cada variável categórica vira um "item" via one-hot encoding (`SG_PARTIDO=PT`, `DS_GENERO=FEMININO`, `cluster=Abastados`, ...).

Variáveis escolhidas:
- `SG_PARTIDO`, `DS_GENERO`, `DS_GRAU_INSTRUCAO`, `DS_ESTADO_CIVIL`, `DS_COR_RACA`, `SG_UF_NASCIMENTO` — perfil do candidato.
- `cluster` — o nome do cluster da Fase 2 (Bisecting K-Means, k=3), recalculado acima. Incluir o cluster como item deixa a pergunta "o que caracteriza cada cluster?" (já respondida via ficha técnica na Fase 2) ser respondida de novo do ponto de vista de regras — e permite achar combinações de 2+ variáveis que sozinhas não bastariam.
- `SG_UF` — só entra na visão Brasil (`UF=None`); com uma UF fixa ela teria um único valor e não discriminaria nada.
- `DS_SIT_TOT_TURNO` (eleito ou não) — só entra depois que a eleição de fato ocorrer (a coluna hoje vem toda com o mesmo valor, `#NULO`). A seção que usa isso fica pronta, mas roda condicionalmente lá na frente.

`DS_OCUPACAO` fica de fora por ora: são 49 categorias diferentes pra só ~200 candidatos — a maioria das ocupações teria 1 ou 2 candidatos, suporte baixo demais pra dizer qualquer coisa. Fica como exercício: agrupem em categorias maiores (ex.: "política", "direito", "saúde", "educação", "empresário"...) e testem.


In [35]:
colunas_apriori = ['SG_PARTIDO', 'DS_GENERO', 'DS_GRAU_INSTRUCAO', 'DS_ESTADO_CIVIL',
                   'DS_COR_RACA', 'SG_UF_NASCIMENTO', 'cluster', 'DS_OCUPACAO']

if UF is None:
    colunas_apriori.append('SG_UF')

# a eleição de 2026 ainda não ocorreu -> DS_SIT_TOT_TURNO vem toda com o mesmo valor por enquanto
usar_resultado_eleicao = df_cluster['DS_SIT_TOT_TURNO'].nunique() > 1
if usar_resultado_eleicao:
    colunas_apriori.append('DS_SIT_TOT_TURNO')

cesta_base = df_cluster[['SQ_CANDIDATO'] + colunas_apriori].copy()
cesta = pd.get_dummies(cesta_base, columns=colunas_apriori, prefix_sep='=')
cesta = cesta.set_index('SQ_CANDIDATO')

print(f"{cesta.shape[0]} candidatos (cestas) x {cesta.shape[1]} itens possíveis (um por valor de cada variável)")
cesta_base.head()


2189 candidatos (cestas) x 217 itens possíveis (um por valor de cada variável)


,SQ_CANDIDATO,SG_PARTIDO,DS_GENERO,DS_GRAU_INSTRUCAO,DS_ESTADO_CIVIL,DS_COR_RACA,SG_UF_NASCIMENTO,cluster,DS_OCUPACAO
0,20002532430,NOVO,FEMININO,SUPERIOR COMPLETO,SOLTEIRO(A),PRETA,AL,Profissionais Diplomados,PSICÓLOGO
1,20002532431,NOVO,MASCULINO,SUPERIOR COMPLETO,CASADO(A),BRANCA,AL,Elite,SERVIDOR PÚBLICO FEDERAL
2,20002532432,NOVO,MASCULINO,SUPERIOR COMPLETO,CASADO(A),BRANCA,AL,Elite,ADVOGADO
3,20002532433,NOVO,FEMININO,SUPERIOR COMPLETO,VIÚVO(A),BRANCA,AL,Elite,SERVIDOR PÚBLICO ESTADUAL
4,20002532434,NOVO,MASCULINO,SUPERIOR COMPLETO,CASADO(A),BRANCA,AL,Elite,"CORRETOR DE IMÓVEIS, SEGUROS, TÍTULOS E VALORES"


## Itens frequentes: o que aparece sozinho, com que frequência

O Apriori funciona em duas etapas. Primeiro, encontra **itemsets frequentes** — combinações de 1, 2, 3... itens que aparecem juntas em pelo menos `min_suporte` das cestas. Só depois, na segunda etapa, ele vira **regras** (`A → B`). Começamos pelos itemsets de um item só — a frequência "crua" de cada valor, antes de cruzar qualquer coisa.


In [36]:
min_suporte = 0.02  # ~4 candidatos; ajuste se quiser regras mais raras (menor) ou mais comuns (maior)

itens_frequentes = apriori(cesta, min_support=min_suporte, use_colnames=True)
print(f"{len(itens_frequentes)} itemsets frequentes (min_suporte={min_suporte})")

individuais = itens_frequentes[itens_frequentes['itemsets'].apply(lambda x: len(x) == 1)].copy()
individuais['item'] = individuais['itemsets'].apply(lambda x: next(iter(x)))
individuais['qtd_candidatos'] = (individuais['support'] * len(cesta)).round().astype(int)
individuais.sort_values('support', ascending=False)[['item', 'support', 'qtd_candidatos']].head(15)


947 itemsets frequentes (min_suporte=0.02)


,item,support,qtd_candidatos
19,DS_GENERO=MASCULINO,0.623572,1365
24,DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,0.583371,1277
26,DS_ESTADO_CIVIL=CASADO(A),0.490635,1074
30,DS_COR_RACA=PARDA,0.472362,1034
43,cluster=Elite,0.439470,962
18,DS_GENERO=FEMININO,0.376428,824
28,DS_ESTADO_CIVIL=SOLTEIRO(A),0.356784,781
29,DS_COR_RACA=BRANCA,0.350845,768
33,SG_UF_NASCIMENTO=BA,0.220192,482
22,DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO,0.218821,479


### Suporte, confiança e lift — o que cada métrica mede

Regras de associação têm o formato **"se A, então B"** (A = antecedente, B = consequente), e três números resumem o quão forte é essa relação:

- **Suporte** — fração de candidatos em que A e B aparecem juntos: `suporte(A→B) = P(A e B)`. Mede o quão comum é a combinação; suporte baixo (poucos candidatos) é terreno fértil pra coincidência.
- **Confiança** — entre os candidatos que têm A, qual fração também tem B: `confiança(A→B) = P(B | A) = suporte(A e B) / suporte(A)`. É a "taxa de acerto" da regra.
- **Lift** — o quanto a confiança da regra é maior (ou menor) do que se A e B fossem independentes: `lift(A→B) = confiança(A→B) / suporte(B)`. `lift = 1` → A e B são independentes (a regra não diz nada); `lift > 1` → aparecem juntos mais do que o esperado ao acaso; `lift < 1` → aparecem juntos menos do que o esperado.

**Um cuidado importante nesta base:** ela tem só ~200 candidatos espalhados por 27 partidos e 27 estados — em média, uns 7 candidatos por partido ou por UF. Com grupos tão pequenos, bastam 3 ou 4 coincidências pra um lift parecer altíssimo, sem que isso signifique relação real nenhuma. Por isso toda tabela de regras abaixo vem com uma coluna `qtd_candidatos` — sempre olhem ela antes de confiar num lift alto.


In [37]:
regras = association_rules(itens_frequentes, metric='lift', min_threshold=1)
regras['qtd_candidatos'] = (regras['support'] * len(cesta)).round().astype(int)
regras = regras.sort_values('confidence', ascending=False)


def formatar_regras(df_regras):
    """Troca antecedents/consequents (frozenset) por texto legível, só pra exibição —
    os dados originais (usados pra filtrar/ordenar) continuam intactos."""
    df_fmt = df_regras.copy()
    df_fmt['antecedents'] = df_fmt['antecedents'].apply(lambda x: ', '.join(sorted(str(i) for i in x)))
    df_fmt['consequents'] = df_fmt['consequents'].apply(lambda x: ', '.join(sorted(str(i) for i in x)))
    return df_fmt


print(f"{len(regras)} regras (lift >= 1)")
formatar_regras(regras[['antecedents', 'consequents', 'qtd_candidatos', 'support', 'confidence', 'lift']].head(10))


4474 regras (lift >= 1)


,antecedents,consequents,qtd_candidatos,support,confidence,lift
3692,"DS_COR_RACA=BRANCA, DS_OCUPACAO=DEPUTADO, cluster=Elite",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,47,0.021471,1.0,1.714174
3736,"DS_COR_RACA=PARDA, SG_UF_NASCIMENTO=PE, cluster=Elite",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,62,0.028323,1.0,1.714174
3722,"DS_COR_RACA=PARDA, SG_UF_NASCIMENTO=MA, cluster=Elite",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,71,0.032435,1.0,1.714174
3650,"DS_COR_RACA=BRANCA, SG_UF_NASCIMENTO=CE, cluster=Elite",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,71,0.032435,1.0,1.714174
3706,"DS_COR_RACA=PARDA, SG_UF_NASCIMENTO=BA, cluster=Elite",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,91,0.041571,1.0,1.714174
3640,"DS_COR_RACA=BRANCA, SG_UF_NASCIMENTO=BA, cluster=Elite",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,62,0.028323,1.0,1.714174
442,"SG_PARTIDO=PDT, cluster=Elite",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,44,0.020101,1.0,1.714174
2174,"DS_ESTADO_CIVIL=CASADO(A), DS_GENERO=FEMININO, cluster=Elite",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,157,0.071722,1.0,1.714174
3744,"DS_COR_RACA=PARDA, DS_OCUPACAO=ADVOGADO, cluster=Elite",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,53,0.024212,1.0,1.714174
2909,"DS_GENERO=MASCULINO, DS_OCUPACAO=MÉDICO, cluster=Elite",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,49,0.022385,1.0,1.714174


### Cuidado: suporte pequeno infla o lift

Vamos ver isso na prática: as regras de maior lift da base inteira, sem nenhum filtro de suporte, comparadas às regras de maior lift **exigindo pelo menos 10 candidatos** por trás delas.


In [38]:
regras_1_consequente = regras[regras['consequents'].apply(lambda x: len(x) == 1)].copy()
colunas_exibir = ['antecedents', 'consequents', 'qtd_candidatos', 'confidence', 'lift']

print("--- Maior lift, sem filtro de suporte ---")
display(formatar_regras(regras_1_consequente.sort_values('lift', ascending=False)[colunas_exibir].head(5)))

print("--- Maior lift, exigindo qtd_candidatos >= 10 ---")
display(formatar_regras(regras_1_consequente[regras_1_consequente['qtd_candidatos'] >= 10]
        .sort_values('lift', ascending=False)[colunas_exibir].head(5)))


--- Maior lift, sem filtro de suporte ---


,antecedents,consequents,qtd_candidatos,confidence,lift
2134,"DS_ESTADO_CIVIL=SOLTEIRO(A), DS_GENERO=FEMININO, DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO",cluster=Base Popular,54,0.658537,4.215019
4328,"DS_COR_RACA=BRANCA, DS_GENERO=MASCULINO, DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, cluster=Elite",DS_OCUPACAO=DEPUTADO,45,0.159011,4.047374
3238,"DS_COR_RACA=BRANCA, DS_GENERO=MASCULINO, cluster=Elite",DS_OCUPACAO=DEPUTADO,45,0.156794,3.990965
668,"DS_GENERO=FEMININO, DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO",cluster=Base Popular,104,0.619048,3.962267
1425,"DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO, DS_OCUPACAO=OUTROS",cluster=Base Popular,62,0.601942,3.852779


--- Maior lift, exigindo qtd_candidatos >= 10 ---


,antecedents,consequents,qtd_candidatos,confidence,lift
2134,"DS_ESTADO_CIVIL=SOLTEIRO(A), DS_GENERO=FEMININO, DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO",cluster=Base Popular,54,0.658537,4.215019
4328,"DS_COR_RACA=BRANCA, DS_GENERO=MASCULINO, DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, cluster=Elite",DS_OCUPACAO=DEPUTADO,45,0.159011,4.047374
3238,"DS_COR_RACA=BRANCA, DS_GENERO=MASCULINO, cluster=Elite",DS_OCUPACAO=DEPUTADO,45,0.156794,3.990965
668,"DS_GENERO=FEMININO, DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO",cluster=Base Popular,104,0.619048,3.962267
1425,"DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO, DS_OCUPACAO=OUTROS",cluster=Base Popular,62,0.601942,3.852779


No primeiro grupo, é comum ver pares como `SG_UF=ES → SG_UF_NASCIMENTO=ES` com lift acima de 30: candidatos a governador costumam concorrer no estado onde nasceram, então UF e UF de nascimento são quase a mesma informação — com só 4 ou 5 candidatos capixabas na base, a coincidência vira um lift enorme sem dizer nada de novo. No segundo grupo, com um piso de candidatos por trás da regra, sobram combinações genuinamente mais interessantes (ex.: escolaridade + gênero prevendo cluster). Essa é a leitura que vamos seguir usando daqui pra frente.

### Filtrando pra regras que valem a pena olhar


In [39]:
suporte_pratico = 0.03    # ~6 candidatos
confianca_pratica = 0.5

regras_simples = regras[
    regras['antecedents'].apply(lambda x: len(x) <= 2) & regras['consequents'].apply(lambda x: len(x) == 1)
]
regras_confiaveis = regras_simples[
    (regras_simples['support'] >= suporte_pratico) & (regras_simples['confidence'] >= confianca_pratica)
]

print(f"{len(regras_confiaveis)} regras com suporte >= {suporte_pratico} e confiança >= {confianca_pratica}")
formatar_regras(regras_confiaveis.sort_values('confidence', ascending=False)[colunas_exibir].head(15))


304 regras com suporte >= 0.03 e confiança >= 0.5


,antecedents,consequents,qtd_candidatos,confidence,lift
1490,"DS_ESTADO_CIVIL=CASADO(A), DS_OCUPACAO=ADVOGADO",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,83,1.000000,1.714174
402,"SG_PARTIDO=MDB, cluster=Elite",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,68,1.000000,1.714174
1694,"DS_OCUPACAO=ADVOGADO, cluster=Elite",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,133,1.000000,1.714174
1664,"SG_UF_NASCIMENTO=MA, cluster=Elite",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,114,1.000000,1.714174
588,"SG_PARTIDO=PT, cluster=Elite",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,69,1.000000,1.714174
215,DS_OCUPACAO=MÉDICO,DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,68,1.000000,1.714174
1542,"DS_ESTADO_CIVIL=SOLTEIRO(A), cluster=Elite",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,238,1.000000,1.714174
726,"DS_GENERO=FEMININO, DS_OCUPACAO=ADVOGADO",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,67,1.000000,1.714174
716,"DS_GENERO=FEMININO, cluster=Elite",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,332,1.000000,1.714174
1650,"SG_UF_NASCIMENTO=BA, cluster=Elite",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,204,0.995122,1.705812


## O que prediz o cluster de um candidato?

Filtramos as regras cujo **consequente é um único item de cluster** — ou seja, "se o candidato tem tais características, ele cai no cluster X". É a mesma pergunta que a Fase 2 respondeu via ficha técnica, agora vista pelo ângulo de combinações de atributos categóricos.

Um cuidado extra aqui: os três clusters do Bisecting K-Means são **bem desbalanceados** (Abastados = 78%, Jovens = 14%, Sem bens e alta escolaridade = 8%). Ordenar só por confiança faria a tabela inteira virar uma lista de regras "→ Abastados" — fácil de acertar por ser a maioria, mas com lift baixo (perto de 1,3, já que acertar o grupo majoritário não é surpresa nenhuma). E ordenar por confiança com um piso fixo (ex.: confiança >= 0.5) teria o efeito oposto: nenhuma regra sustentaria isso para "Sem bens e alta escolaridade" (só 16 candidatos — a maior confiança que qualquer regra de até 2 itens alcança pra esse cluster é 0.36). Por isso pegamos as regras de **maior lift dentro de cada cluster**, separadamente — assim os três aparecem, e dá pra comparar a força relativa de cada um.


In [40]:
def regras_para_consequente(regras_completas, prefixo, n_por_valor=6, min_suporte=min_suporte, max_antecedentes=2):
    """
    Entre as regras cujo único item consequente comece com `prefixo` (ex.: 'cluster='),
    devolve as `n_por_valor` de MAIOR LIFT pra CADA valor distinto do consequente — sem piso
    de confiança fixo, porque um valor raro (poucos candidatos) não sustenta regras de
    confiança alta mesmo quando o lift é forte; ordenar só por confiança deixaria os valores
    mais comuns dominando a tabela inteira (ver célula anterior).
    """
    candidatas = regras_completas[
        regras_completas['consequents'].apply(lambda x: len(x) == 1 and next(iter(x)).startswith(prefixo))
        & regras_completas['antecedents'].apply(lambda x: len(x) <= max_antecedentes)
        & (regras_completas['support'] >= min_suporte)
    ].copy()
    candidatas['alvo'] = candidatas['consequents'].apply(lambda x: next(iter(x)))
    resultado = (candidatas.sort_values('lift', ascending=False)
                            .groupby('alvo', group_keys=False)
                            .head(n_por_valor)
                            .sort_values(['alvo', 'lift'], ascending=[True, False]))
    return resultado.drop(columns='alvo')


regras_cluster = regras_para_consequente(regras, 'cluster=')
print(f"{len(regras_cluster)} regras (até 6 por cluster, ordenadas por lift dentro de cada um)")
formatar_regras(regras_cluster[colunas_exibir])


24 regras (até 6 por cluster, ordenadas por lift dentro de cada um)


,antecedents,consequents,qtd_candidatos,confidence,lift
668,"DS_GENERO=FEMININO, DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO",cluster=Base Popular,104,0.619048,3.962267
1425,"DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO, DS_OCUPACAO=OUTROS",cluster=Base Popular,62,0.601942,3.852779
1352,"DS_ESTADO_CIVIL=SOLTEIRO(A), DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO",cluster=Base Popular,117,0.582090,3.725714
1401,"DS_COR_RACA=PRETA, DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO",cluster=Base Popular,52,0.525253,3.361923
1412,"DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO, SG_UF_NASCIMENTO=BA",cluster=Base Popular,60,0.512821,3.282351
180,DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO,cluster=Base Popular,240,0.501044,3.206974
1702,"DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, DS_OCUPACAO=DEPUTADO",cluster=Elite,71,0.972603,2.213126
590,"DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, SG_PARTIDO=PT",cluster=Elite,69,0.920000,2.093430
510,"DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, SG_PARTIDO=PP",cluster=Elite,44,0.916667,2.085845
1872,"DS_ESTADO_CIVIL=CASADO(A), DS_OCUPACAO=MÉDICO",cluster=Elite,46,0.901961,2.052383


### Visualizando como grafo

Uma tabela com dezenas de regras é difícil de escanear — um grafo interativo ajuda a ver de relance quais itens aparecem mais como "explicação" (antecedentes) e quais aparecem mais como "resultado" (consequentes). Cada item vira um nó (`SG_PARTIDO=PCO`, `cluster=D`, ...); cada regra vira uma seta do antecedente pro consequente, com espessura proporcional à confiança. Os nós que começam com `cluster=` ficam destacados em vermelho e maiores, pra serem fáceis de achar no meio dos outros itens. Passe o mouse sobre uma seta pra ver confiança, lift e quantos candidatos sustentam aquela regra — e arraste os nós se a rede ficar embaralhada.


In [41]:
def grafo_regras_html(regras_plot, arquivo_html, titulo, prefixo_destaque='cluster='):
    """
    Desenha as regras como uma rede interativa (pyvis) e grava em `arquivo_html`. Cada REGRA
    vira um nó de antecedente e um nó de consequente (quando o antecedente tem mais de um
    item, eles aparecem juntos no mesmo nó, separados por "+") ligados por UMA única aresta
    — nunca uma aresta por item isolado, pra não sugerir que um item sozinho sustenta a
    confiança/lift de uma regra que na verdade depende da combinação inteira (era isso que
    causava setas duplicadas entre o mesmo par de nós). Nós que contêm algum item começando
    com `prefixo_destaque` (por padrão, os clusters da Fase 2) ficam destacados em vermelho e
    maiores. Espessura da aresta = confiança da regra; passe o mouse pra ver confiança, lift
    e quantos candidatos a sustentam. Os controles de física (embaixo do grafo) deixam
    ajustar a repulsão/gravidade entre os nós ao vivo.

    Retorna um IFrame carregando o arquivo — se aparecer em branco/preto aqui embaixo (alguns
    ambientes bloqueiam o JavaScript de saídas de célula em notebooks "não confiáveis"), abra
    `arquivo_html` direto no navegador; o grafo é o mesmo.
    """
    print(titulo)
    net = Network(height='600px', width='100%', bgcolor='#222222', font_color='white',
                  notebook=True, directed=True, cdn_resources='in_line')

    nos_adicionados = set()
    for _, row in regras_plot.iterrows():
        antecedente = ' + '.join(sorted(str(item) for item in row['antecedents']))
        consequente = ' + '.join(sorted(str(item) for item in row['consequents']))

        for label, itemset in [(antecedente, row['antecedents']), (consequente, row['consequents'])]:
            if label in nos_adicionados:
                continue
            destaque = any(str(item).startswith(prefixo_destaque) for item in itemset)
            net.add_node(
                label, label, title=label,
                color='#e41a1c' if destaque else '#1f78b4',
                size=30 if destaque else 12,
                font={'size': 22 if destaque else 14},
            )
            nos_adicionados.add(label)

        titulo_aresta = (f"Confiança: {row['confidence']:.2f} | Lift: {row['lift']:.2f} | "
                          f"{row['qtd_candidatos']} candidatos")
        net.add_edge(antecedente, consequente, value=row['confidence'], title=titulo_aresta)

    net.show_buttons(filter_=['physics'])
    net.save_graph(arquivo_html)
    print(f"Grafo salvo em: {arquivo_html} — se não aparecer abaixo, abra esse arquivo no navegador.")
    return IFrame(arquivo_html, width='100%', height=750)


grafo_regras_html(regras_cluster.head(20), 'grafo_regras_cluster.html', 'O que prediz o cluster de um candidato?')


O que prediz o cluster de um candidato?


Grafo salvo em: grafo_regras_cluster.html — se não aparecer abaixo, abra esse arquivo no navegador.


## O que caracteriza quem já é de um cluster?

A seção anterior fixou o **consequente** (`cluster=X`) e perguntou o que o prediz. Agora invertemos: fixamos o **antecedente** — "dado que o candidato é do cluster X, ..." — e olhamos pra que outros itens aparecem como consequente com lift alto. É uma pergunta diferente, não a mesma coisa ao contrário: uma regra `A → B` de lift alto não garante que `B → A` também tenha lift alto (confiança e suporte de cada lado são diferentes), então vale olhar as duas direções separadamente.


In [42]:
def regras_para_antecedente(regras_completas, item_fixo, n=8, min_suporte=min_suporte, max_consequentes=1):
    """
    Entre as regras cujo antecedente é EXATAMENTE {item_fixo} (um único item), devolve as
    `n` de maior lift. Direção oposta de `regras_para_consequente`: lá fixamos o "efeito"
    (o consequente) e procurávamos o que o prediz; aqui fixamos a "causa" (o antecedente) e
    olhamos o que ela prediz.
    """
    candidatas = regras_completas[
        regras_completas['antecedents'].apply(lambda x: len(x) == 1 and next(iter(x)) == item_fixo)
        & regras_completas['consequents'].apply(lambda x: len(x) <= max_consequentes)
        & (regras_completas['support'] >= min_suporte)
    ]
    return candidatas.sort_values('lift', ascending=False).head(n)


regras_por_cluster_fixo = {
    nome: regras_para_antecedente(regras, nome)
    for nome in ('cluster=Abastados', 'cluster=Jovens', 'cluster=Sem bens e alta escolaridade')
}

for item_fixo, subset in regras_por_cluster_fixo.items():
    print(f"--- {item_fixo} ({len(subset)} regras) ---")
    display(formatar_regras(subset[colunas_exibir]))


--- cluster=Abastados (0 regras) ---


,antecedents,consequents,qtd_candidatos,confidence,lift


--- cluster=Jovens (0 regras) ---


,antecedents,consequents,qtd_candidatos,confidence,lift


--- cluster=Sem bens e alta escolaridade (0 regras) ---


,antecedents,consequents,qtd_candidatos,confidence,lift


### Visualizando como grafo (antecedente fixo)

Mesmo grafo interativo de antes, só que agora as três "causas" (os clusters) são o ponto de partida fixo, e os itens que aparecem como consequência é que variam.


In [43]:
regras_antecedente_fixo = pd.concat(regras_por_cluster_fixo.values())
grafo_regras_html(regras_antecedente_fixo, 'grafo_regras_antecedente_fixo.html', 'O que caracteriza quem já é de um cluster?')


O que caracteriza quem já é de um cluster?
Grafo salvo em: grafo_regras_antecedente_fixo.html — se não aparecer abaixo, abra esse arquivo no navegador.


## E quando a eleição já tiver ocorrido?

Esta seção só roda depois que a Fase 0 for rodada de novo com o resultado oficial (`DS_SIT_TOT_TURNO` deixa de vir toda com o mesmo valor `#NULO`) — por enquanto, ela se anuncia e não faz nada. Repare que não precisamos escrever nenhum código novo: como `DS_SIT_TOT_TURNO` já entrou na cesta lá na preparação (quando aplicável), a mesma `regras_para_consequente` e o mesmo `grafo_regras_html` servem pra essa pergunta também.


In [44]:
if usar_resultado_eleicao:
    regras_eleicao = regras_para_consequente(regras, 'DS_SIT_TOT_TURNO=')
    print(f"{len(regras_eleicao)} regras apontando para o resultado da eleição")
    display(formatar_regras(regras_eleicao[colunas_exibir].head(15)))
    grafo_regras_html(regras_eleicao.head(20), 'grafo_regras_eleicao.html', 'O que prediz ser eleito?', prefixo_destaque='DS_SIT_TOT_TURNO=')
else:
    print("DS_SIT_TOT_TURNO ainda não tem informação real (a eleição de 2026 não ocorreu) — "
          "essa seção fica pronta pra usar assim que a Fase 0 for rodada de novo com o resultado oficial. "
          "Por enquanto, pulamos.")


DS_SIT_TOT_TURNO ainda não tem informação real (a eleição de 2026 não ocorreu) — essa seção fica pronta pra usar assim que a Fase 0 for rodada de novo com o resultado oficial. Por enquanto, pulamos.


---
## Fechamento e próximos passos

Resumindo o roteiro: transformamos cada candidato numa "cesta" de atributos categóricos (partido, gênero, escolaridade, cor/raça, UF de nascimento, e o cluster da Fase 2), minerar itemsets frequentes com o Apriori, geramos regras `A → B` com suporte/confiança/lift, e aprendemos — na prática, não só na teoria — que suporte baixo infla lift artificialmente numa base pequena como esta.

**Exercícios sugeridos:**
- Agrupar `DS_OCUPACAO` em categorias maiores e incluir na cesta.
- Variar `min_suporte` (célula da seção "Itens frequentes") e ver como o número de regras muda.
- Replicar para `SENADOR` ou `DEPUTADO FEDERAL`, na sua UF — só muda a célula de configuração, do mesmo jeito que nas fases anteriores.
- Depois que a eleição ocorrer: rodar a Fase 0 de novo, e conferir se a seção "E quando a eleição já tiver ocorrido?" acima encontra regras fortes ligadas a `DS_SIT_TOT_TURNO=ELEITO`.

Próxima etapa do pipeline: `04_deteccao_anomalias.ipynb`.
